# CAD — Cognitive Authenticity Detection
## Demo Notebook
This notebook demonstrates the full CAD pipeline: rule-based baseline, TF-IDF model, and GUS scoring.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import pandas as pd
import numpy as np
from src.utils import load_data, split_data, extract_linguistic_features, compute_gus, verdict_from_gus
from src.model import RuleBasedCAD, TFIDFModel, BayesianConfidenceWrapper

print('Imports OK')

In [ ]:
# Load and inspect data
df = load_data('../data/answers.csv')
print(f'Dataset: {len(df)} rows')
print(df['Label'].value_counts())
df.head(6)

In [ ]:
# Rule-based baseline
rule_model = RuleBasedCAD()

for _, row in df.head(6).iterrows():
    label, prob = rule_model.predict(row['Answer'])
    print(f"[{row['Label']}] -> [{label}] (prob={prob:.2f}) | {row['Answer'][:60]}...")

In [ ]:
# TF-IDF model
from src.utils import label_encode

train_df, test_df = split_data(df, test_size=0.2)

train_texts = train_df['Answer'].tolist()
train_labels = label_encode(train_df['Label'])
train_ling = [extract_linguistic_features(t) for t in train_texts]

tfidf_model = TFIDFModel()
tfidf_model.fit(train_texts, train_labels, train_ling)
print('TF-IDF model trained.')

In [ ]:
# Evaluate on test set
from sklearn.metrics import classification_report

test_texts = test_df['Answer'].tolist()
test_ling = [extract_linguistic_features(t) for t in test_texts]
test_labels = label_encode(test_df['Label'])

probs = [tfidf_model.predict_proba_single(t, l) for t, l in zip(test_texts, test_ling)]
preds = [1 if p >= 0.5 else 0 for p in probs]

print(classification_report(test_labels, preds, target_names=['Shallow', 'Deep']))

In [ ]:
# GUS scoring with Bayesian calibration
bayesian = BayesianConfidenceWrapper(temperature=1.5)

test_answers = [
    "Photosynthesis converts light energy into glucose via the Calvin cycle in chloroplasts.",
    "Plants use sunlight to make food.",
    "Gravity is the force of attraction between masses, described by F = Gm1m2/r^2.",
    "Gravity pulls things down.",
]

print(f"{'Answer':<55} {'Raw Prob':>9} {'Calib':>7} {'GUS':>5} {'Verdict'}")
print('-' * 105)
for ans in test_answers:
    ling = extract_linguistic_features(ans)
    raw_prob = tfidf_model.predict_proba_single(ans, ling)
    cal_prob, uncertainty, anomalies = bayesian.score(raw_prob, ling)
    gus = compute_gus(cal_prob, ling)
    verdict = verdict_from_gus(gus)
    print(f"{ans[:53]:<55} {raw_prob:>9.3f} {cal_prob:>7.3f} {gus:>5} {verdict}")

In [ ]:
# Visualise GUS distribution on dataset
import matplotlib.pyplot as plt

all_gus = []
for _, row in df.iterrows():
    ling = extract_linguistic_features(row['Answer'])
    raw = tfidf_model.predict_proba_single(row['Answer'], ling)
    cal, _, _ = bayesian.score(raw, ling)
    gus = compute_gus(cal, ling)
    all_gus.append({'gus': gus, 'label': row['Label']})

gus_df = pd.DataFrame(all_gus)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for label, color in [('Deep', '#2D6B4A'), ('Shallow', '#7A3B1E')]:
    subset = gus_df[gus_df['label'] == label]['gus']
    ax1.hist(subset, bins=10, alpha=0.7, label=label, color=color)
ax1.set_xlabel('GUS Score'); ax1.set_ylabel('Count'); ax1.set_title('GUS Distribution by Label'); ax1.legend()

ax2.boxplot([gus_df[gus_df['label']=='Deep']['gus'], gus_df[gus_df['label']=='Shallow']['gus']], labels=['Deep', 'Shallow'])
ax2.set_ylabel('GUS Score'); ax2.set_title('GUS Box Plot by Label')
plt.tight_layout(); plt.savefig('../results/gus_distribution.png', dpi=150); plt.show()
print('Saved results/gus_distribution.png')